In [1]:
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import matplotlib.pyplot as plt
from pyspark.sql.types import *
from pyspark.sql import DataFrame, SparkSession
from pyspark.ml.feature import Tokenizer, StopWordsRemover
import unidecode
import seaborn as sns
import pandas as pd

In [ ]:
spark = (
    SparkSession.builder
      .config("spark.driver.memory", "48g")                 # ajuste p/ 16g–24g
      .config("spark.sql.execution.arrow.pyspark.enabled", "true")
      .config("spark.sql.execution.arrow.maxRecordsPerBatch", "20000")
      .config("spark.sql.files.maxPartitionBytes", 64 * 1024 * 1024)  # 64MB/partição
      .config("spark.driver.maxResultSize", "0")            # sem limite de resultado (cuidado)
      .getOrCreate()
)

spark.version

'4.0.1'

In [3]:
# read table
data = spark.read.option("header", True).parquet(
    "C:/Users/clamo/Documents/Doutorado/HC/hc_models/dados/evolucoes.parquet"
)

data.count()

922813

In [4]:
# filter pacientes in target data
pacientes_target = spark.read.option("header", True).parquet(
    "C:/Users/clamo/Documents/Doutorado/HC/hc_models/hc_ufpe_longstay/sample_target_internacao.parquet"
).select("prontuario").distinct()

data = data.join(pacientes_target, on="prontuario", how="inner")
data.count()

614386

In [5]:
# count nulls in evolucao_text
data.filter(F.col("descricao").isNull()).count()

1602

In [6]:
data.printSchema()

root
 |-- prontuario: long (nullable = true)
 |-- atendimento: long (nullable = true)
 |-- codigo_paciente: long (nullable = true)
 |-- sexo: string (nullable = true)
 |-- descricao: string (nullable = true)
 |-- unidade_funcional: string (nullable = true)
 |-- numero_quarto: string (nullable = true)
 |-- data_internacao: string (nullable = true)
 |-- data_saida: string (nullable = true)
 |-- data_obito: string (nullable = true)
 |-- data_criacao: string (nullable = true)
 |-- data_alteracao: string (nullable = true)
 |-- profissional: string (nullable = true)
 |-- tipo_qualificacao: string (nullable = true)
 |-- sigla_conselho: string (nullable = true)
 |-- conselho: string (nullable = true)
 |-- registro_conselho: string (nullable = true)
 |-- cbo_principal: string (nullable = true)
 |-- cartao_sus_cns: double (nullable = true)
 |-- ind_pendente: string (nullable = true)



In [7]:
data.select("atendimento", "descricao").show(5, truncate=False)

+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [8]:
data.filter(
    ~F.col("data_saida").rlike(r'^\s*\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\s*$')
).select("data_saida").distinct().show(30, False)

+----------+
|data_saida|
+----------+
+----------+



In [9]:
# filtrar quem esta com data de internação em formato correto
data = data.filter(
    F.col("data_saida").rlike(
        r'^\s*\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}(\.\d{3})?\s*$'
    )
)
data.count()

610320

In [10]:
@F.pandas_udf("string")
def normalize_text_udf(s: pd.Series) -> pd.Series:
    # preserva NULL; aplica só em não nulos
    s2 = s.str.lower()
    return s2.where(s2.isna(), s2.map(unidecode))


def remove_stopwords_spark(df: DataFrame, input_col: str, output_col: str = "clean_text") -> DataFrame:
    """
    Tokenizes and removes stopwords from a text column in a PySpark DataFrame.

    Args:
        df (DataFrame): Input DataFrame.
        input_col (str): Name of the column containing raw text.
        output_col (str): Name of the final cleaned text column (joined tokens).

    Returns:
        DataFrame: Original DataFrame with added columns:
                   - 'tokens': tokenized words
                   - 'filtered_tokens': after stopword removal
                   - output_col: cleaned text as single string
    """
    tokenizer = Tokenizer(inputCol=input_col, outputCol="tokens")
    df_tokenized = tokenizer.transform(df)

    remover = StopWordsRemover(
        inputCol="tokens",
        outputCol="filtered_tokens",
        stopWords=StopWordsRemover.loadDefaultStopWords("portuguese")
    )
    df_filtered = remover.transform(df_tokenized)

    df_cleaned = df_filtered.withColumn(output_col, F.concat_ws(" ", "filtered_tokens"))

    columns_to_drop = ["tokens", "filtered_tokens"]
    df_cleaned = df_cleaned.drop(*columns_to_drop)

    return df_cleaned

In [11]:
def remove_acentos(col):
    # mapeamento 1-para-1 (mesmo tamanho nas duas strings)
    src = "áàãâäÁÀÃÂÄéèêëÉÈÊËíìîïÍÌÎÏóòõôöÓÒÕÔÖúùûüÚÙÛÜçÇñÑ"
    dst = "aaaaaAAAAAeeeeEEEEiiiiIIIIoooooOOOOOuuuuUUUUcCnN"
    return F.translate(col, src, dst)

In [12]:
data_processed = (data
    # fill null values in specified columns
    .na.fill({
        "descricao": ""
    })
    # normalize and clean text column
    .withColumn("descricao_processed", 
        F.trim(F.lower(F.col("descricao")))
    )

)

#data_processed = data_processed.withColumn(
#    "descricao_processed",
#    remove_acentos(F.col("descricao"))
#)


# to string type
data_processed = (data_processed
    .withColumn("descricao_processed", F.col("descricao_processed").cast(StringType()))
)

# remove special characters
data_processed = data_processed.withColumn(
    "descricao_processed",
    F.regexp_replace(F.col("descricao_processed"), r"[^a-zA-Z0-9\s]", " ")
)

# remove stopwords
data_processed = remove_stopwords_spark(df=data_processed, 
                                            input_col='descricao_processed', 
                                            output_col='descricao_processed')

# Remove números
data_processed = data_processed.withColumn(
    "descricao_processed",
    F.regexp_replace("descricao_processed", "[0-9]", "")
)

# Remove espaços duplicados
data_processed = data_processed.withColumn(
    "descricao_processed",
    F.regexp_replace("descricao_processed", "\\s+", " ")
)

In [13]:
# converte as strings para timestamp
data_processed = data_processed.withColumn("data_saida", F.to_timestamp("data_saida")) 

In [14]:
# filter cases above august 2022
data_processed = data_processed.filter(F.col("data_saida") >= F.lit("2022-08-01"))

In [15]:
data_processed.printSchema()

root
 |-- prontuario: long (nullable = true)
 |-- atendimento: long (nullable = true)
 |-- codigo_paciente: long (nullable = true)
 |-- sexo: string (nullable = true)
 |-- descricao: string (nullable = false)
 |-- unidade_funcional: string (nullable = true)
 |-- numero_quarto: string (nullable = true)
 |-- data_internacao: string (nullable = true)
 |-- data_saida: timestamp (nullable = true)
 |-- data_obito: string (nullable = true)
 |-- data_criacao: string (nullable = true)
 |-- data_alteracao: string (nullable = true)
 |-- profissional: string (nullable = true)
 |-- tipo_qualificacao: string (nullable = true)
 |-- sigla_conselho: string (nullable = true)
 |-- conselho: string (nullable = true)
 |-- registro_conselho: string (nullable = true)
 |-- cbo_principal: string (nullable = true)
 |-- cartao_sus_cns: double (nullable = true)
 |-- ind_pendente: string (nullable = true)
 |-- descricao_processed: string (nullable = false)



In [16]:
#data_processed_out = data_processed.select("prontuario", "data_saida", "descricao", "descricao_processed").toPandas()

In [17]:
#data_processed_out.to_parquet("sample_target_internacao.parquet")
#data_processed.show(5)

In [18]:
# construir meses de referencia a partir das datas minima e maxima
bounds = data.agg(
    F.date_trunc("month", F.min("data_saida")).alias("min_m"),
    F.date_trunc("month", F.max("data_saida")).alias("max_m"),
)

ref_dates_data = bounds.select(
    F.expr("sequence(min_m, max_m, interval 1 month) as ref_dates")
).select(F.explode("ref_dates").alias("ref_date"))

# lista de ref dates distintos
ref_dates = [r.ref_date for r in ref_dates_data.collect()]
len(ref_dates)

48

In [19]:
# define dataframe para incorporar dados ao cursor
evolucao = spark.createDataFrame([], schema=StructType())

# para cada data de referencia, filtrar os dados anteriores a ela, para calculo das variaveis
for ref_ts in ref_dates:

    data_filtered = data_processed.filter(
        (F.col("data_saida") < F.lit(ref_ts)) &
        (F.col("data_saida") >= F.add_months(F.lit(ref_ts), -12))
    )

    # concatenar as evolucoes do paciente
    evolucao_ref_date = data_filtered.groupBy("prontuario").agg(
        F.concat_ws(" ", F.collect_list("descricao_processed")).alias("descricao_unica")
    )

    evolucao_ref_date = evolucao_ref_date.withColumn("date_ref", F.lit(ref_ts).cast("timestamp"))

    evolucao = evolucao.unionByName(evolucao_ref_date, allowMissingColumns=True)

    print(ref_ts)

2021-10-01 00:00:00
2021-11-01 00:00:00
2021-12-01 00:00:00
2022-01-01 00:00:00
2022-02-01 00:00:00
2022-03-01 00:00:00
2022-04-01 00:00:00
2022-05-01 00:00:00
2022-06-01 00:00:00
2022-07-01 00:00:00
2022-08-01 00:00:00
2022-09-01 00:00:00
2022-10-01 00:00:00
2022-11-01 00:00:00
2022-12-01 00:00:00
2023-01-01 00:00:00
2023-02-01 00:00:00
2023-03-01 00:00:00
2023-04-01 00:00:00
2023-05-01 00:00:00
2023-06-01 00:00:00
2023-07-01 00:00:00
2023-08-01 00:00:00
2023-09-01 00:00:00
2023-10-01 00:00:00
2023-11-01 00:00:00
2023-12-01 00:00:00
2024-01-01 00:00:00
2024-02-01 00:00:00
2024-03-01 00:00:00
2024-04-01 00:00:00
2024-05-01 00:00:00
2024-06-01 00:00:00
2024-07-01 00:00:00
2024-08-01 00:00:00
2024-09-01 00:00:00
2024-10-01 00:00:00
2024-11-01 00:00:00
2024-12-01 00:00:00
2025-01-01 00:00:00
2025-02-01 00:00:00
2025-03-01 00:00:00
2025-04-01 00:00:00
2025-05-01 00:00:00
2025-06-01 00:00:00
2025-07-01 00:00:00
2025-08-01 00:00:00
2025-09-01 00:00:00


In [20]:
# filter pacientes e date_ref in target data
pacientes_target = spark.read.option("header", True).parquet(
    "C:/Users/clamo/Documents/Doutorado/HC/hc_models/features/sample_target_internacao.parquet"
).select("prontuario", "date_ref").distinct()

pacientes_target.count()

23855

In [21]:
evolucao = evolucao.join(pacientes_target, on=["prontuario", "date_ref"], how="inner")
evolucao.count()

6651

In [22]:
evolpd = evolucao.toPandas()

In [23]:
evolpd.to_parquet("C:/Users/clamo/Documents/Doutorado/HC/hc_models/features/sample_embbeding_evolucao.parquet")